# VQE 教程：顶层接口 + 手动 Parameter-Shift 梯度下降

本教程分两部分：
1. 顶层：直接使用 `VQERunner.run_model` 跑 VQE
2. 拆解：手动实现 `parameter-shift` 梯度 + 梯度下降更新

默认用 `Simulator`，便于稳定复现与快速实验。

In [1]:
import numpy as np

from quantum_hw import QuantumHardwareClient, VQERunner, QuantumCircuit
from quantum_hw.api import Backend
from quantum_hw.algorithms.vqe import (
    build_ising_hamiltonian,
    _evaluate_energy_with_backend,
    _parameter_shift_gradient,
)
from quantum_hw.algorithms.ansatz_templates import build_hardware_efficient_ansatz, build_hardware_efficient_ansatz_symbolic

def section(title: str):
    print("\n" + "=" * 18, title, "=" * 18)

## 1) 问题设置（Ising 哈密顿量）

In [2]:
section("setup")
num_qubits = 2
layers = 1
shots = 2048
max_iters = 15
learning_rate = 0.2
shift = np.pi / 2
seed = 42




================== setup ==================


## 2) 顶层调用：`VQERunner.run_model`

直接把优化细节交给框架。

In [3]:
section("high-level VQE")
client = QuantumHardwareClient()
vqe = VQERunner(
    client=client,
    layers=layers,
    shots=shots,
    max_iters=max_iters,
    learning_rate=learning_rate,
    shift=shift,
    zne=False,
    readout_mitigation=False,
    seed=seed,
)

result_top = vqe.run_model(
    name="tutorial_vqe_top",
    num_qubits=num_qubits,
    model="ising",
    model_params={"j": 1.0, "h": 1.0},
    prefer_chips="Simulator",
)

print("best_energy:", result_top.best_energy)
print("best_params (first 6):", result_top.best_params[:6])
print("energy_history:", result_top.energy_history)


================== high-level VQE ==================
[vqe] prepare run: name=tutorial_vqe_top num_qubits=2 provider=quafu model=ising layers=1 shots=2048 max_iters=15
[vqe] candidate chips: ['Simulator']
[vqe] running on chip: Simulator
[vqe] start optimization: iters=15 layers=1 params=8 ansatz=hardwareefficient shots=2048 shift=1.5707963267948966 gradient=parameter-shift
[vqe] iter 0 start
[vqe] iter 0 energy=0.791016 grad_norm=1.898899
[vqe] iter 0 new best energy=0.791016
[vqe] iter 1 start
[vqe] iter 1 energy=-0.138672 grad_norm=1.954821
[vqe] iter 1 new best energy=-0.138672
[vqe] iter 2 start
[vqe] iter 2 energy=-0.851562 grad_norm=1.139151
[vqe] iter 2 new best energy=-0.851562
[vqe] iter 3 start
[vqe] iter 3 energy=-1.070312 grad_norm=0.491431
[vqe] iter 3 new best energy=-1.070312
[vqe] iter 4 start
[vqe] iter 4 energy=-0.902344 grad_norm=1.177139
[vqe] iter 5 start
[vqe] iter 5 energy=-0.776367 grad_norm=1.502030
[vqe] iter 6 start
[vqe] iter 6 energy=-0.854492 grad_norm=1.

## 3) 拆解版：手动写能量评估 + Parameter-Shift 梯度

思路：
- 给定参数 `params` 构造 ansatz 线路
- 用 `client.run_auto(... observables=...)` 得到每个 Pauli 项期望
- 按哈密顿量线性组合得到能量
- 用 parameter-shift 公式算梯度

In [4]:
section("manual energy + gradient")
client_manual = QuantumHardwareClient()

def evaluate_energy(qc: QuantumCircuit, run_name: str, num_qubits: int, backend: Backend, shots: int, observables): # demo code for manual energy evaluation, also see _evaluate_energy_with_backend()
    result = client_manual._run_with_backend(
    qc,
    run_name,
    num_qubits,
    backend=backend,
    chip_name="Simulator",
    shots=shots,
    observables=observables,
    transpile=False,
    )
    exp_map = result.observable_values
    energy = float(sum(coeff * exp_map.get(obs, 0.0) for coeff, obs in hamiltonian))
    return energy, exp_map

def parameter_shift_grad(params: np.ndarray, shift_value: float, run_name: str, num_qubits: int, backend: Backend, shots: int, observables, layers: int): # demo code for manual gradient evaluation using parameter-shift rule, also see _parameter_shift_gradient()
    grads = np.zeros_like(params, dtype=float)
    for i in range(params.size):
        p_plus = params.copy()
        p_minus = params.copy()
        p_plus[i] += shift_value
        p_minus[i] -= shift_value
        qc_plus = build_hardware_efficient_ansatz(num_qubits, p_plus, layers=layers)
        qc_minus = build_hardware_efficient_ansatz(num_qubits, p_minus, layers=layers)
        e_plus, _ = evaluate_energy(qc_plus, f"{run_name}_grad_p{i}", num_qubits, backend, shots, observables)
        e_minus, _ = evaluate_energy(qc_minus, f"{run_name}_grad_m{i}", num_qubits, backend, shots, observables)
        grads[i] = 0.5 * (e_plus - e_minus)
    return grads

# 构造Hamiltonian和ansatz
hamiltonian = build_ising_hamiltonian(num_qubits=num_qubits, j=1.0, h=1.0)
observables = [obs for _, obs in hamiltonian]
print("hamiltonian terms:")
for coeff, obs in hamiltonian:
    print(f"  {coeff:+.3f} * {obs}")

num_params = 2 * num_qubits * (layers + 1)
rng = np.random.default_rng(seed)
params0 = rng.uniform(0.0, 2.0 * np.pi, size=num_params)
qc = build_hardware_efficient_ansatz(num_qubits, params0, layers=layers)
backend = Backend("Simulator")
# 调用demo函数
e0, exp0 = evaluate_energy(qc, "tutorial_vqe_manual_init", num_qubits, backend, shots, observables)
g0 = parameter_shift_grad(params0, shift, "tutorial_vqe", num_qubits, backend, shots, observables, layers)
print("initial energy:", e0)
print("initial grad norm:", float(np.linalg.norm(g0)))
print("initial expectations:", exp0)



================== manual energy + gradient ==================
hamiltonian terms:
  -1.000 * Z0 Z1
  -1.000 * X0
  -1.000 * X1
initial energy: 0.791015625
initial grad norm: 1.898899163520376
initial expectations: {'Z0 Z1': -0.2861328125, 'X0': -0.23828125, 'X1': -0.2666015625}


## 4) 手动梯度下降循环

这里用最朴素的 GD 更新：
$$\theta \leftarrow \theta - \eta \nabla E(\theta)$$

In [5]:
section("manual gradient descent")
params = params0.copy()
energy_history_manual = []

for it in range(max_iters):
    qc = build_hardware_efficient_ansatz(num_qubits, params, layers=layers)
    energy, _ = evaluate_energy(qc, f"tutorial_vqe_manual_iter{it}", num_qubits, backend, shots, observables)
    grads = parameter_shift_grad(params, shift_value=shift, run_name=f"tutorial_vqe_manual_iter{it}", num_qubits=num_qubits, backend=backend, shots=shots, observables=observables, layers=layers)
    grad_norm = float(np.linalg.norm(grads))

    energy_history_manual.append(float(energy))
    print(f"iter={it:02d} energy={energy:.6f} grad_norm={grad_norm:.6f}")

    params = params - learning_rate * grads

energy_final, exp_final = evaluate_energy(qc, run_name="tutorial_vqe_manual_final", num_qubits=num_qubits, backend=backend, shots=shots, observables=observables)
print("manual final energy:", energy_final)
print("manual final expectations:", exp_final)


================== manual gradient descent ==================
iter=00 energy=0.791016 grad_norm=1.898899
iter=01 energy=0.016602 grad_norm=2.089526
iter=02 energy=-0.783203 grad_norm=1.752902
iter=03 energy=-1.312500 grad_norm=1.133422
iter=04 energy=-1.532227 grad_norm=0.765830
iter=05 energy=-1.630859 grad_norm=0.656235
iter=06 energy=-1.730469 grad_norm=0.638366
iter=07 energy=-1.799805 grad_norm=0.605012
iter=08 energy=-1.875977 grad_norm=0.589092
iter=09 energy=-1.931641 grad_norm=0.559208
iter=10 energy=-1.991211 grad_norm=0.530540
iter=11 energy=-2.048828 grad_norm=0.490679
iter=12 energy=-2.076172 grad_norm=0.447716
iter=13 energy=-2.112305 grad_norm=0.401562
iter=14 energy=-2.140625 grad_norm=0.355783
manual final energy: -2.140625
manual final expectations: {'Z0 Z1': 0.4912109375, 'X0': 0.8037109375, 'X1': 0.845703125}


## 5) 顶层与手动版本对比

In [6]:
section("compare")
print("top best energy:", result_top.best_energy)
print("manual final energy:", energy_final)
print("top history:", result_top.energy_history)
print("manual history:", energy_history_manual)


================== compare ==================
top best energy: -2.19140625
manual final energy: -2.140625
top history: [0.791015625, -0.138671875, -0.8515625, -1.0703125, -0.90234375, -0.7763671875, -0.8544921875, -1.083984375, -1.404296875, -1.681640625, -1.890625, -1.9892578125, -2.091796875, -2.1494140625, -2.19140625]
manual history: [0.791015625, 0.0166015625, -0.783203125, -1.3125, -1.5322265625, -1.630859375, -1.73046875, -1.7998046875, -1.8759765625, -1.931640625, -1.9912109375, -2.048828125, -2.076171875, -2.1123046875, -2.140625]


# 6) 自动微分
在设计算法时，可使用自动微分

In [7]:
section("auto-grad VQE")
client = QuantumHardwareClient()
vqe = VQERunner(
    client=client,
    layers=layers,
    shots=shots,
    max_iters=max_iters,
    learning_rate=learning_rate,
    shift=shift,
    zne=False,
    readout_mitigation=False,
    seed=seed,
    gradient_method="autograd",
)

result_top = vqe.run_model(
    name="tutorial_vqe_top",
    num_qubits=num_qubits,
    model="ising",
    model_params={"j": 1.0, "h": 1.0},
    prefer_chips="Simulator",
)

print("best_energy:", result_top.best_energy)
print("best_params (first 6):", result_top.best_params[:6])
print("energy_history:", result_top.energy_history)


================== auto-grad VQE ==================
[vqe] prepare run: name=tutorial_vqe_top num_qubits=2 provider=quafu model=ising layers=1 shots=2048 max_iters=15
[vqe] candidate chips: ['Simulator']
[vqe] running on chip: Simulator
[vqe] start optimization: iters=15 layers=1 params=8 ansatz=hardwareefficient shots=2048 shift=1.5707963267948966 gradient=autograd
[vqe] iter 0 start
[vqe] iter 0 energy=0.821374 grad_norm=1.900223
[vqe] iter 0 new best energy=0.821374
[vqe] iter 1 start
[vqe] iter 1 energy=-0.112317 grad_norm=1.977676
[vqe] iter 1 new best energy=-0.112317
[vqe] iter 2 start
[vqe] iter 2 energy=-0.849452 grad_norm=1.167440
[vqe] iter 2 new best energy=-0.849452
[vqe] iter 3 start
[vqe] iter 3 energy=-1.059629 grad_norm=0.501613
[vqe] iter 3 new best energy=-1.059629
[vqe] iter 4 start
[vqe] iter 4 energy=-0.875138 grad_norm=1.173433
[vqe] iter 5 start
[vqe] iter 5 energy=-0.747315 grad_norm=1.492015
[vqe] iter 6 start
[vqe] iter 6 energy=-0.820599 grad_norm=1.543481
[

# 7) 真实硬件完整梯度下降

下面演示真实硬件路径的完整流程：

In [ ]:
section("high-level VQE")
client = QuantumHardwareClient()
vqe = VQERunner(
    client=client,
    layers=layers,
    shots=shots,
    max_iters=max_iters,
    learning_rate=learning_rate,
    shift=shift,
    zne=True,
    readout_mitigation=True,
    seed=seed,
)

result_top = vqe.run_model(
    name="tutorial_vqe_top",
    num_qubits=num_qubits,
    model="ising",
    model_params={"j": 1.0, "h": 1.0},
    prefer_chips="Baihua",
)

print("best_energy:", result_top.best_energy)
print("best_params (first 6):", result_top.best_params[:6])
print("energy_history:", result_top.energy_history)

## 8) 下一步建议

- 把 `num_qubits` 增加到 4 或 6，观察参数量和收敛速度。
- 把手动 GD 改成 Adam（对应库内默认优化器）。